# Métrica de distância e tipo de votação

## Importando bibliotecas e dados

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.knn_utils import criar_rodar, criar_cv_f1

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

In [2]:
train_df = pd.read_csv("../data/processed/train.csv", index_col="id")
test_df = pd.read_csv("../data/processed/test.csv", index_col="id")
X_train, y_train = train_df.drop(columns="classe"), train_df["classe"]
X_test, y_test = test_df.drop(columns="classe"), test_df["classe"]

rodar = criar_rodar(X_train, X_test, y_train, y_test)
cv_f1 = criar_cv_f1(X_train, y_train)

k_teste = pd.read_csv("../data/processed/k_teste.csv", index_col="k")
k_cv = pd.read_csv("../data/processed/k_cv.csv", index_col="k")
COLS = ["f1_treino", "f1_teste", "gap", "acc_teste", "prec_teste", "rec_teste"]
NOMES = ["F1 treino", "F1 teste", "Gap", "Acurácia teste", "Precision teste", "Recall teste"]

## Investigação da métrica de distância

Com `MinMaxScaler` e k = 15, comparamos Manhattan (p = 1) e Euclidiana (p = 2) na família de Minkowski.

In [3]:
res_dist = pd.DataFrame({"Manhattan (p=1)": rodar("minmax", k=15, p=1),
                         "Euclidiana (p=2)": k_teste.loc[15, COLS].to_dict()}).T[COLS]

tab_dist = res_dist.copy()
tab_dist.columns = NOMES
tab_dist

,F1 treino,F1 teste,Gap,Acurácia teste,Precision teste,Recall teste
Manhattan (p=1),0.8274,0.7995,0.0279,0.8000,0.8013,0.7993
Euclidiana (p=2),0.8230,0.8187,0.0042,0.8194,0.8214,0.8185


In [4]:
cv_dist = pd.DataFrame({"Manhattan (p=1)": cv_f1("minmax", 15, p=1),
                        "Euclidiana (p=2)": tuple(k_cv.loc[15])},
                       index=["F1 médio (cv)", "desvio (cv)"]).T
cv_dist

,F1 médio (cv),desvio (cv)
Manhattan (p=1),0.7950,0.0298
Euclidiana (p=2),0.7821,0.0230


### Resultado da investigação

* No teste, a Euclidiana teve F1 maior (0,819 vs. 0,800) e menor gap. 
* Na validação cruzada, a Manhattan foi melhor (0,795 vs. 0,782). 
* A Euclidiana penaliza mais diferenças grandes; a Manhattan soma as diferenças de forma linear.
* Como os resultados variam, não dá para dizer que uma é superior, Mantivemos a Euclidiana pelo melhor equilíbrio entre treino e teste.


## Votação uniforme e votação ponderada

Com `MinMaxScaler`, k = 15 e p = 2, comparamos `weights="uniform"` e `weights="distance"`.

In [5]:
res_pesos = pd.DataFrame({"uniform": res_dist.loc["Euclidiana (p=2)"].to_dict(),
                          "distance": rodar("minmax", k=15, p=2, pesos="distance")}).T[COLS]
res_pesos.to_csv("../data/processed/votacao.csv", index_label="pesos")

tab_pesos = res_pesos.copy()
tab_pesos.columns = NOMES
tab_pesos

,F1 treino,F1 teste,Gap,Acurácia teste,Precision teste,Recall teste
uniform,0.8230,0.8187,0.0042,0.8194,0.8214,0.8185
distance,1.0000,0.8253,0.1747,0.8258,0.8273,0.8251


In [6]:
cv_pesos = pd.DataFrame({"uniform": tuple(k_cv.loc[15]),
                         "distance": cv_f1("minmax", 15, 2, "distance")},
                        index=["F1 médio (cv)", "desvio (cv)"]).T
cv_pesos.to_csv("../data/processed/votacao_cv.csv", index_label="pesos")
cv_pesos

,F1 médio (cv),desvio (cv)
uniform,0.7821,0.0230
distance,0.7821,0.0250


### Resultado das votações
* Uniforme: todos os k vizinhos têm o mesmo peso. Ponderada: os vizinhos mais próximos têm mais peso.
* A ponderada dá mais influência a um vizinho muito próximo.
* No teste, a diferença foi pequena: 0,825 vs. 0,819. Na validação cruzada, ambas ficaram em 0,782.
* A ponderada chegou a 1,000 no treino porque cada ponto é vizinho de si mesmo. Por isso, ficamos com a uniforme, que é mais simples e equilibrada.
